# Cap-and-Trade Simulation — ECON7720 Lecture 04

**Interactive simulation** showing how a permit market delivers cost-effective abatement.

- **Left panel**: Individual firm MAC curves. Circles (●) = post-trade allocation; Crosses (×) = uniform standard.
- **Centre panel**: Aggregate MAC meets the cap (vertical) → the permit price emerges.
- **Right panel**: Total abatement cost — trading vs uniform standard.

Use the sliders to explore how the **cap level**, **number of firms**, and **MAC heterogeneity** affect the permit price and cost savings.

In [ ]:
%pip install -q ipywidgets matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider
from IPython.display import display

# ── UQ branding ──────────────────────────────────────────────────────
UQ_PURPLE = "#512478"
UQ_CYAN = "#0099CC"
COLORS = [UQ_PURPLE, UQ_CYAN, "#E6308A", "#2EA836", "#E87722", "#00A4BD",
          "#8B5CF6", "#DC2626"]

# ── Model ────────────────────────────────────────────────────────────
# Each firm i has MAC_i(a_i) = slope_i * a_i  (linear, through origin)
# a_i = abatement = E0 - e_i;  E0 = uncontrolled emissions per firm

E0_PER_FIRM = 10.0


def make_slopes(n, heterogeneity):
    """Generate n MAC slopes with given heterogeneity (spread)."""
    if n == 1:
        return np.array([2.0])
    base = 2.0
    spread = np.linspace(-heterogeneity, heterogeneity, n)
    return np.clip(base + spread, 0.3, 10.0)


def solve_trading(slopes, cap):
    """Cost-effective (equimarginal) allocation under trading."""
    n = len(slopes)
    total_abatement = max(n * E0_PER_FIRM - cap, 0.0)
    inv_slopes = 1.0 / slopes
    permit_price = total_abatement / np.sum(inv_slopes)
    a_star = permit_price / slopes
    cost_trading = 0.5 * slopes * a_star**2
    return permit_price, a_star, cost_trading


def solve_uniform(slopes, cap):
    """Uniform standard: each firm abates equally."""
    n = len(slopes)
    total_abatement = max(n * E0_PER_FIRM - cap, 0.0)
    a_uniform = total_abatement / n
    cost_uniform = 0.5 * slopes * a_uniform**2
    return a_uniform, cost_uniform

In [ ]:
def plot_simulation(cap=24, firms=4, heterogeneity=1.5):
    """Draw the three-panel cap-and-trade simulation."""
    n = int(firms)
    slopes = make_slopes(n, heterogeneity)
    permit_price, a_star, cost_trading = solve_trading(slopes, cap)
    a_uniform, cost_uniform = solve_uniform(slopes, cap)
    total_E0 = n * E0_PER_FIRM

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("white")
    plt.subplots_adjust(wspace=0.32)
    fig.suptitle("Cap-and-Trade Simulation", fontsize=14,
                 fontweight="bold", color=UQ_PURPLE, y=1.02)

    # ── Panel 1: Individual firms ────────────────────────────────────
    ax1 = axes[0]
    ax1.set_title("Individual firms", fontsize=11, color=UQ_PURPLE)
    ax1.set_xlabel("Abatement $a_i$")
    ax1.set_ylabel("$ / unit")

    a_max = E0_PER_FIRM * 1.1
    a_range = np.linspace(0, a_max, 200)

    for i in range(n):
        c = COLORS[i % len(COLORS)]
        ax1.plot(a_range, slopes[i] * a_range, color=c, lw=1.5, alpha=0.7,
                 label=f"Firm {i+1}")
        # Post-trade (circle)
        ax1.plot(a_star[i], permit_price, "o", color=c, ms=7, zorder=5)
        # Uniform (cross)
        ax1.plot(a_uniform, slopes[i] * a_uniform, "x", color=c,
                 ms=7, mew=2, zorder=5)

    ax1.axhline(permit_price, color=UQ_PURPLE, ls="--", lw=1, alpha=0.6,
                label=f"$p$ = {permit_price:.1f}")
    ax1.set_xlim(0, a_max)
    ax1.set_ylim(0, max(permit_price * 2.5, 5))
    ax1.legend(fontsize=7, loc="upper left", framealpha=0.8)

    # ── Panel 2: Permit market ───────────────────────────────────────
    ax2 = axes[1]
    ax2.set_title("Permit market", fontsize=11, color=UQ_PURPLE)
    ax2.set_xlabel("Emissions $E$")
    ax2.set_ylabel("$ / unit")

    inv_sum = np.sum(1.0 / slopes)
    E_range = np.linspace(0, total_E0, 300)
    agg_mac = np.maximum((total_E0 - E_range) / inv_sum, 0)

    ax2.plot(E_range, agg_mac, color=UQ_PURPLE, lw=2.5, label="Aggregate MAC")
    ax2.axvline(cap, color=UQ_CYAN, lw=2.5, label=f"Cap = {cap:.0f}")
    ax2.plot([0, cap], [permit_price, permit_price], "--",
             color=UQ_PURPLE, lw=1, alpha=0.6)
    ax2.plot(cap, permit_price, "o", color=UQ_PURPLE, ms=8, zorder=5)
    ax2.annotate(f"$p$ = {permit_price:.1f}", xy=(cap, permit_price),
                 xytext=(cap + total_E0 * 0.05, permit_price + 0.5),
                 fontsize=9, color=UQ_PURPLE,
                 arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1))
    ax2.set_xlim(0, total_E0 * 1.05)
    ax2.set_ylim(0, max(agg_mac[0] * 1.2, 5))
    ax2.legend(fontsize=8, loc="upper right", framealpha=0.8)

    # ── Panel 3: Cost comparison ─────────────────────────────────────
    ax3 = axes[2]
    ax3.set_title("Total abatement cost", fontsize=11, color=UQ_PURPLE)

    tc_trade = np.sum(cost_trading)
    tc_uniform = np.sum(cost_uniform)
    saving = tc_uniform - tc_trade
    saving_pct = 100 * saving / tc_uniform if tc_uniform > 0 else 0

    bars = ax3.bar(["Uniform\nstandard", "Cap-and-\ntrade"],
                   [tc_uniform, tc_trade],
                   color=[UQ_PURPLE, UQ_CYAN], width=0.5,
                   edgecolor="white", linewidth=1.5)

    if saving > 0.01:
        ax3.annotate(f"Saving: ${saving:.0f}\n({saving_pct:.0f}%)",
                     xy=(1, tc_trade),
                     xytext=(1.35, (tc_uniform + tc_trade) / 2),
                     fontsize=9, color=UQ_PURPLE, ha="center",
                     fontweight="bold",
                     arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1.2))

    ax3.set_ylabel("Total cost ($)")
    ax3.set_ylim(0, max(tc_uniform * 1.4, 1))

    for bar, val in zip(bars, [tc_uniform, tc_trade]):
        ax3.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.5,
                 f"${val:.0f}", ha="center", va="bottom",
                 fontsize=9, fontweight="bold", color=UQ_PURPLE)

    plt.tight_layout()
    plt.show()

In [ ]:
interact(
    plot_simulation,
    cap=IntSlider(value=24, min=5, max=75, step=1,
                  description="Cap (total E):",
                  style={"description_width": "initial"}),
    firms=IntSlider(value=4, min=2, max=8, step=1,
                    description="Number of firms:",
                    style={"description_width": "initial"}),
    heterogeneity=FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1,
                              description="MAC heterogeneity:",
                              style={"description_width": "initial"}),
);

## Things to try

1. **Tighten the cap** (slide left): watch the permit price rise and costs increase — but trading always beats the uniform standard.
2. **Set heterogeneity to 0**: all firms are identical → trading saves nothing (no gains from trade).
3. **Increase heterogeneity**: the cost gap widens — more diverse firms = bigger gains from trade.
4. **Add more firms**: more sources to exploit → the market becomes more efficient.
5. **Set cap = total E₀** (n × 10): no abatement needed → price = 0, costs = 0.

---
*ECON7720 — Ecological & Environmental Economics | The University of Queensland | Dr Juan Soto-Diaz*